# MINI Cells — Experiment 006: 10M Language Scaling

This notebook trains `minicells-v2` and a parameter-matched Transformer from random initialization through 10M consumed tokens. With two T4 GPUs the two models run concurrently, one process per GPU.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')
if not (ROOT / '.git').exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=ROOT, check=True)
    subprocess.run(['git', 'switch', 'main'], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    raise RuntimeError('Experiment 006 requires CUDA')


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_language_bridge.py', 'tests/test_language_ablation.py', 'tests/test_language_scaling.py', '-q'], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/run_consumer_language_scaling.py'], cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import Image, Markdown, display
OUT = ROOT / 'results' / 'consumer-language-scaling-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
display(Markdown(f"**10M PPL ratio:** {decision['comparison']['ppl_ratio_10m']:.4f}×  \
**Slope ratio:** {decision['comparison']['slope_ratio_to_transformer']:.4f}"))
for name in ['ppl-scaling.png', 'nll-scaling.png', 'relative-gap.png', 'throughput.png']:
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'generation-progression.md').read_text(encoding='utf-8')))


In [ ]:
# Set to True only after reviewing the outputs above.
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable, 'scripts/publish_experiment_006_results.py', '--push'], cwd=ROOT, check=True)
